<h1> Lab 5-2 Topic Modeling</h1>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssuai/Hands-On-Large-Language-Models/blob/main/chapter05/lab_5-2.ipynb)

### [OPTIONAL] - Installing Packages on Colab

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [1]:
%%capture
!pip uninstall -y transformers numpy pyarrow pandas bertopic datasets
!rm -rf /root/.cache/huggingface/  # cache migration issue

!pip install "numpy<2.0.0" "pyarrow==14.0.1" "pandas<2.2.0"
!pip install transformers==4.44.2  # EncoderDecoderCache
!pip install datasets bertopic datamapplot

# **ArXiv Articles: Computation and Language**

In [2]:
# Load data from huggingface
from datasets import load_dataset
dataset = load_dataset("maartengr/arxiv_nlp")["train"]

# Extract metadata
abstracts = list(dataset["Abstracts"])
titles = list(dataset["Titles"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/datasets/utils/file_utils.py:1250: FutureWarning: The 'verbose' keyword in pd.read_csv is deprecated and will be removed in a future version.
  return pd.read_csv(xopen(filepath_or_buffer, "rb", download_config=download_config), **kwargs)


# **A Common Pipeline for Text Clustering**

In [3]:
from sentence_transformers import SentenceTransformer

# Create an embedding for each abstract
embedding_model = SentenceTransformer('thenlper/gte-small')
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


In [4]:
from umap import UMAP

# We reduce the input embeddings from 384 dimenions to 5 dimenions
umap_model = UMAP(
    n_components=5, min_dist=0.0, metric='cosine', random_state=42
)
reduced_embeddings = umap_model.fit_transform(embeddings)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [5]:
from hdbscan import HDBSCAN

# We fit the model and extract the clusters
hdbscan_model = HDBSCAN(
    min_cluster_size=50, metric='euclidean', cluster_selection_method='eom'
).fit(reduced_embeddings)
clusters = hdbscan_model.labels_

# How many clusters did we generate?
len(set(clusters))

158

In [6]:
# import numpy as np

In [7]:
# import pandas as pd

In [8]:
# import matplotlib.pyplot as plt

# From Text Clustering to Topic Modeling

## **BERTopic: A Modular Topic Modeling Framework**

In [9]:
# !pip install pyarrow==14.0.1
# import pyarrow
# pyarrow.__version__

In [10]:
from bertopic import BERTopic

# Train our model with our previously defined models
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
).fit(abstracts, embeddings)

2026-04-04 18:03:53,975 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-04 18:04:46,156 - BERTopic - Dimensionality - Completed ✓
2026-04-04 18:04:46,158 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-04 18:04:48,320 - BERTopic - Cluster - Completed ✓
2026-04-04 18:04:48,333 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-04 18:04:52,269 - BERTopic - Representation - Completed ✓


Now, let's start exploring the topics that we got by running the code above.

In [11]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,15072,-1_the_of_and_to,"[the, of, and, to, in, we, for, that, language...",[ Machine Learning has been the quintessentia...
1,0,2261,0_question_answer_questions_qa,"[question, answer, questions, qa, answering, a...",[ With the rise of large-scale pre-trained la...
2,1,2088,1_speech_asr_recognition_end,"[speech, asr, recognition, end, acoustic, spea...","[ Voice Assistants such as Alexa, Siri, and G..."
3,2,901,2_summarization_summaries_summary_abstractive,"[summarization, summaries, summary, abstractiv...",[ Pre-trained neural abstractive summarizatio...
4,3,840,3_hate_offensive_speech_detection,"[hate, offensive, speech, detection, toxic, so...",[ Hate speech detection has become a hot topi...
...,...,...,...,...,...
153,152,53,152_coherence_discourse_paragraph_text,"[coherence, discourse, paragraph, text, cohesi...",[ While there has been significant progress t...
154,153,52,153_backdoor_attacks_attack_triggers,"[backdoor, attacks, attack, triggers, poisoned...",[ Deep neural networks (DNNs) and natural lan...
155,154,52,154_counterfactual_counterfactuals_cad_causal,"[counterfactual, counterfactuals, cad, causal,...","[ For text classification tasks, finetuned la..."
156,155,50,155_opinion_reviews_summaries_summarization,"[opinion, reviews, summaries, summarization, r...",[ When faced with a large number of product r...


Hundreds of topics were generated using the default model! To get the top 10 keywords per topic as well as their c-TF-IDF weights, we can use the `get_topic()` function:

In [12]:
topic_model.get_topic(0)

[('question', np.float64(0.02098829190198092)),
 ('answer', np.float64(0.01566775886686886)),
 ('questions', np.float64(0.015607949552734037)),
 ('qa', np.float64(0.015553973901637605)),
 ('answering', np.float64(0.01467417625982838)),
 ('answers', np.float64(0.009717685398370396)),
 ('retrieval', np.float64(0.009226821172955417)),
 ('comprehension', np.float64(0.00760552518955503)),
 ('reading', np.float64(0.0070425298773680025)),
 ('knowledge', np.float64(0.006218269459020677))]

We can use the `find_topics()` function to search for specific topics based on a search term. Let’s search for a topic about topic modeling:

In [13]:
topic_model.find_topics("topic modeling")

([23, -1, 53, 36, 75],
 [np.float32(0.9544884),
  np.float32(0.91206205),
  np.float32(0.90722215),
  np.float32(0.9043106),
  np.float32(0.9037067)])

It returns that topic 22 has a relatively high similarity (0.95) with our search term. If we then inspect the topic, we can see that it is indeed a topic about topic modeling:

In [14]:
topic_model.get_topic(22)

[('translation', np.float64(0.031404748566652194)),
 ('mt', np.float64(0.028979860119650532)),
 ('qe', np.float64(0.0238763000124614)),
 ('quality', np.float64(0.018782324385224796)),
 ('translations', np.float64(0.016892366436611073)),
 ('machine', np.float64(0.015550554961506398)),
 ('metrics', np.float64(0.014220710561005762)),
 ('evaluation', np.float64(0.013604706795997071)),
 ('human', np.float64(0.012468142456656861)),
 ('estimation', np.float64(0.01153337232523286))]

That seems like a topic that is, in part, characterized by the classic LDA technique. Let's see if the BERTopic paper was also assigned to topic 22:

In [15]:
topic_model.topics_[titles.index('BERTopic: Neural topic modeling with a class-based TF-IDF procedure')]

23

It is! We expected it might be because there are non-LDA specific words in the topic describtion such as "clustering" and "topic".

### **Visualizations**

**Visualize Documents**

In [16]:
# Visualize topics and documents
fig = topic_model.visualize_documents(
    titles,
    reduced_embeddings=reduced_embeddings,
    width=1200,
    hide_annotations=True
)

# Update fonts of legend for easier visualization
fig.update_layout(font=dict(size=16))

In [17]:
# Visualize barchart with ranked keywords
topic_model.visualize_barchart()

# Visualize relationships between topics
topic_model.visualize_heatmap(n_clusters=30)

# Visualize the potential hierarchical structure of topics
topic_model.visualize_hierarchy()

## **Representation Models**

In these examples that follow, we will update our topic representations **after** having trained our model. This allows for quick iteration. If, however, you want to use a representation model at the start of training, you will need to run it as follows:

```python
from bertopic.representation import KeyBERTInspired
from bertopic import BERTopic

# Create your representation model
representation_model = KeyBERTInspired()

# Use the representation model in BERTopic on top of the default pipeline
topic_model = BERTopic(representation_model=representation_model)
```

To use the representation models, we are first going to duplicate our topic model such that easily show the differences between a model with and without representation model.

In [18]:
# Save original representations
from copy import deepcopy
original_topics = deepcopy(topic_model.topic_representations_)

In [19]:
import pandas as pd
def topic_differences(model, original_topics, nr_topics=5):
    """Show the differences in topic representations between two models """
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    for topic in range(nr_topics):

        # Extract top 5 words per topic per model
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])
        df.loc[len(df)] = [topic, og_words, new_words]

    return df

### KeyBERTInspired

In [20]:
from bertopic.representation import KeyBERTInspired

# Update our topic representations using KeyBERTInspired
representation_model = KeyBERTInspired()
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

,Topic,Original,Updated
0,0,question | answer | questions | qa | answering,answering | questions | comprehension | answer...
1,1,speech | asr | recognition | end | acoustic,phonetic | speech | language | transcription |...
2,2,summarization | summaries | summary | abstract...,summarization | summarizers | summaries | summ...
3,3,hate | offensive | speech | detection | toxic,hate | hateful | harassment | language | offen...
4,4,relation | extraction | re | relations | entity,relation | relations | relational | extracting...


### Maximal Marginal Relevance

In [21]:
from bertopic.representation import MaximalMarginalRelevance

# Update our topic representations to MaximalMarginalRelevance
representation_model = MaximalMarginalRelevance(diversity=0.5)
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

,Topic,Original,Updated
0,0,question | answer | questions | qa | answering,questions | retrieval | comprehension | knowle...
1,1,speech | asr | recognition | end | acoustic,speech | asr | error | model | automatic
2,2,summarization | summaries | summary | abstract...,summarization | document | extractive | rouge ...
3,3,hate | offensive | speech | detection | toxic,hate | toxic | social | platforms | dataset
4,4,relation | extraction | re | relations | entity,extraction | relations | entity | level | distant


## Text Generation



### Flan-T5

In [22]:
from transformers import pipeline
from bertopic.representation import TextGeneration

prompt = """I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the documents and keywords, what is this topic about?"""

# Update our topic representations using Flan-T5
generator = pipeline('text2text-generation', model='google/flan-t5-small')
representation_model = TextGeneration(
    generator, prompt=prompt, doc_length=50, tokenizer="whitespace"
)
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
100%|██████████| 158/158 [01:22<00:00,  1.92it/s]


,Topic,Original,Updated
0,0,question | answer | questions | qa | answering,Question Answering | | | |
1,1,speech | asr | recognition | end | acoustic,Speech-to-text translation | | | |
2,2,summarization | summaries | summary | abstract...,Summarization | | | |
3,3,hate | offensive | speech | detection | toxic,hate speech detection | | | |
4,4,relation | extraction | re | relations | entity,relation extraction | | | |


### OpenAI

In [23]:
import openai
from bertopic.representation import OpenAI

prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following format:
topic: <short topic label>
"""

# Update our topic representations using GPT-3.5
client = openai.OpenAI(api_key="YOUR_KEY_HERE")
representation_model = OpenAI(
    client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt
)
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

  0%|          | 0/158 [00:00<?, ?it/s]


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_KEY*HERE. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
# Visualize topics and documents
fig = topic_model.visualize_document_datamap(
    titles,
    topics=list(range(20)),
    reduced_embeddings=reduced_embeddings,
    width=1200,
    label_font_size=11,
    label_wrap_width=20,
    use_medoids=True,
)
plt.savefig("datamapplot.png", dpi=300)


## **BONUS**: Word Cloud

Make sure to pip install `wordcloud` first in order to follow this bonus:


First, we need to make sure that each topic is described by a bit more words than just 10 as that would make for a much more interesting wordcloud.

In [ ]:
topic_model.update_topics(abstracts, top_n_words=500)

Then, we can run the following code to generate the wordcloud for our topic modeling topic:

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def create_wordcloud(model, topic):
    plt.figure(figsize=(10,5))
    text = {word: value for word, value in model.get_topic(topic)}
    wc = WordCloud(background_color="white", max_words=1000, width=1600, height=800)
    wc.generate_from_frequencies(text)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.show()

# Show wordcloud
create_wordcloud(topic_model, topic=17)